# SubsiSmart KZ: исследовательский ноутбук

Цель: понять, какие заявки выигрывают/проигрывают, добавить data understanding, корреляции, explainability и подготовить расширение датасета новыми источниками.

## 1. Настройка окружения и загрузка исходных данных
Импорт библиотек, фиксирование random seed, загрузка базового датасета и определение целевой переменной.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

from data_pipeline import load_combined_raw_data
from scoring_engine import SubsidyScoring

SEED = 42
np.random.seed(SEED)

raw_df = load_combined_raw_data('data')
scoring = SubsidyScoring()
df = scoring.preprocess_data(raw_df)
df, _ = scoring.create_kmeans_segments(df, n_clusters=4)
df = scoring.detect_anomalies(df)
df = scoring.calculate_risk_score(df)

target_col = 'is_rejected'
df[target_col] = (df['Статус заявки'].astype(str) == 'Отклонена').astype(int)

print('Rows:', len(df), 'Cols:', len(df.columns))
df.head(3)

## 2. Data Understanding: структура, пропуски, классы и качество

In [ ]:
summary = scoring.data_understanding_summary(df)

print('Shape:', summary['shape'])
print('\nTop missing:')
print(pd.Series(summary['missing_by_column']).head(10))
print('\nClass balance (is_rejected):')
print(df[target_col].value_counts(normalize=True).rename('share'))

duplicates = int(df.duplicated(subset=['Номер заявки']).sum()) if 'Номер заявки' in df.columns else int(df.duplicated().sum())
print('\nDuplicates:', duplicates)

num_cols = ['Норматив', 'Причитающая сумма', 'Превышение', 'Risk_Score']
available = [c for c in num_cols if c in df.columns]
display(df[available].describe().T)

quality_flags = {
    'negative_amounts': int((df['Причитающая сумма'] < 0).sum()) if 'Причитающая сумма' in df.columns else 0,
    'negative_normativ': int((df['Норматив'] < 0).sum()) if 'Норматив' in df.columns else 0,
    'zero_amount_nonzero_normativ': int(((df['Причитающая сумма'] == 0) & (df['Норматив'] > 0)).sum()) if set(['Причитающая сумма', 'Норматив']).issubset(df.columns) else 0,
}
print('\nQuality flags:', quality_flags)

## 3. EDA по заявкам и субсидиям: кто выигрывает и при каких условиях

## 4. Корреляционная матрица и проверка мультиколлинеарности

## 5. Фичи для причин победы/отклонения

## 6. Бейзлайн-модель для прогноза решения по заявке

In [ ]:
sns.set_theme(style='whitegrid')

# Win-rate по регионам и направлениям
region_win = (
    df.assign(is_win=(df['Статус заявки'].astype(str).isin(['Исполнена', 'Одобрена'])).astype(int))
      .groupby('Область', as_index=False)['is_win'].mean()
      .sort_values('is_win', ascending=False)
      .head(15)
)

direction_win = (
    df.assign(is_win=(df['Статус заявки'].astype(str).isin(['Исполнена', 'Одобрена'])).astype(int))
      .groupby('Направление водства', as_index=False)['is_win'].mean()
      .sort_values('is_win', ascending=False)
      .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=region_win, x='is_win', y='Область', ax=axes[0], palette='Greens_r')
axes[0].set_title('Top win-rate по регионам')
axes[0].set_xlabel('Win-rate')

sns.barplot(data=direction_win, x='is_win', y='Направление водства', ax=axes[1], palette='Blues_r')
axes[1].set_title('Top win-rate по направлениям')
axes[1].set_xlabel('Win-rate')

plt.tight_layout()
plt.show()

# Распределение сумм по статусам
plt.figure(figsize=(10, 5))
plot_df = df[df['Причитающая сумма'].notna()].copy()
plot_df['log_amount'] = np.log1p(plot_df['Причитающая сумма'].clip(lower=0))
sns.boxplot(data=plot_df, x='Статус заявки', y='log_amount')
plt.xticks(rotation=25)
plt.title('log(1+Причитающая сумма) по статусам')
plt.show()

In [ ]:
# Корреляции
corr = pd.DataFrame(scoring.correlation_matrix(df))
if not corr.empty:
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
    plt.title('Correlation matrix (Pearson)')
    plt.show()

# Spearman
num_for_corr = [c for c in ['Норматив', 'Причитающая сумма', 'Месяц', 'День_недели', 'Час', 'Квартал', 'Превышение', 'Risk_Score'] if c in df.columns]
if len(num_for_corr) > 1:
    spearman_corr = df[num_for_corr].corr(method='spearman').round(3)
    display(spearman_corr)

# VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_cols = [c for c in ['Норматив', 'Причитающая сумма', 'Месяц', 'День_недели', 'Час', 'Квартал', 'Превышение'] if c in df.columns]
if len(vif_cols) >= 2:
    X_vif = df[vif_cols].fillna(df[vif_cols].median(numeric_only=True))
    vif_df = pd.DataFrame({
        'feature': vif_cols,
        'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    }).sort_values('VIF', ascending=False)
    display(vif_df)
    print('Признаки с VIF > 10 желательно пересмотреть/трансформировать.')

In [ ]:
# Feature pipeline
candidate_features = [
    'Норматив', 'Причитающая сумма', 'Месяц', 'День_недели', 'Час', 'Квартал', 'Превышение',
    'Область', 'Направление водства', 'Статус заявки', 'Risk_Level', 'Кластер'
]
features = [c for c in candidate_features if c in df.columns]

X = df[features].copy()
y = df[target_col].copy()

num_features = X.select_dtypes(include=['number']).columns.tolist()
cat_features = [c for c in X.columns if c not in num_features]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ]
)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp)

models = {
    'log_reg': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED),
    'rf': RandomForestClassifier(n_estimators=300, max_depth=None, random_state=SEED, class_weight='balanced_subsample')
}

results = []
fitted = {}
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_valid)[:, 1]
    pred = (proba >= 0.5).astype(int)

    results.append({
        'model': name,
        'roc_auc': roc_auc_score(y_valid, proba),
        'pr_auc': average_precision_score(y_valid, proba),
        'f1': f1_score(y_valid, pred),
        'recall_rejected': recall_score(y_valid, pred)
    })
    fitted[name] = pipe

results_df = pd.DataFrame(results).sort_values('pr_auc', ascending=False)
display(results_df)

best_name = results_df.iloc[0]['model']
best_model = fitted[best_name]
print('Best model:', best_name)

test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)
print(classification_report(y_test, test_pred, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, test_pred))

## 7. Объяснимость модели: глобальные и локальные причины (SHAP)
## 8. Генератор текстового объяснения "почему отклонено" (AI-assistant style)

In [ ]:
def explain_with_template(shap_values_row, feature_names, top_n=5):
    pairs = sorted(zip(feature_names, shap_values_row), key=lambda x: abs(x[1]), reverse=True)[:top_n]

    negative = [p for p in pairs if p[1] > 0]  # вклад в класс rejected
    positive = [p for p in pairs if p[1] < 0]

    reasons = [f"{name}: вклад {value:.3f}" for name, value in negative]
    improve = [f"Улучшить фактор '{name}'" for name, _ in positive[:3]]

    return {
        'main_reasons': reasons if reasons else ['Сильных негативных факторов не найдено'],
        'what_to_improve': improve if improve else ['Проверить полноту и корректность полей заявки']
    }

if SHAP_AVAILABLE:
    # SHAP для лучшей модели
    prep = best_model.named_steps['preprocessor']
    model = best_model.named_steps['model']
    X_valid_t = prep.transform(X_valid)

    # Для больших данных берем сэмпл
    sample_idx = np.random.choice(np.arange(X_valid_t.shape[0]), size=min(300, X_valid_t.shape[0]), replace=False)
    X_sample = X_valid_t[sample_idx]

    try:
        explainer = shap.Explainer(model, X_sample)
        shap_values = explainer(X_sample)

        shap.plots.bar(shap_values, max_display=15)

        feature_names = prep.get_feature_names_out()
        one_explanation = explain_with_template(shap_values.values[0], feature_names)
        print('AI-style explanation sample:')
        print(one_explanation)
    except Exception as ex:
        print('SHAP не удалось построить:', ex)
else:
    print('Пакет shap не установлен. Установите: pip install shap')

# Правило-ориентированное объяснение на уровне заявки
sample_ids = df['Номер заявки'].astype(str).head(3).tolist() if 'Номер заявки' in df.columns else []
for app_id in sample_ids:
    print('\nApplication ID:', app_id)
    print(scoring.explain_application_by_id(df, app_id))

## 9. Добавление новых данных (CSV/API) и инкрементальное обновление датасета
## 10. Расширение датасета из новых источников и контроль качества
## 11. Переобучение и сравнение метрик после расширения данных

In [ ]:
def map_to_standard_schema(df_new: pd.DataFrame) -> pd.DataFrame:
    """Пример маппинга колонок внешних источников к внутренней схеме."""
    rename_map = {
        'region': 'Область',
        'direction': 'Направление водства',
        'status': 'Статус заявки',
        'normative': 'Норматив',
        'amount': 'Причитающая сумма',
        'application_id': 'Номер заявки',
        'submitted_at': 'Дата поступления'
    }
    cols = {k: v for k, v in rename_map.items() if k in df_new.columns}
    return df_new.rename(columns=cols)


def incremental_merge(base_df: pd.DataFrame, new_df: pd.DataFrame) -> pd.DataFrame:
    """Merge + дедупликация по Номер заявки (последняя версия побеждает)."""
    if 'Номер заявки' not in base_df.columns or 'Номер заявки' not in new_df.columns:
        merged = pd.concat([base_df, new_df], ignore_index=True, sort=False)
        return merged.drop_duplicates()

    merged = pd.concat([base_df, new_df], ignore_index=True, sort=False)
    merged = merged.drop_duplicates(subset=['Номер заявки'], keep='last')
    return merged


# Пример: загрузка новых CSV из data/new_sources
new_sources_dir = Path('data/new_sources')
new_sources_dir.mkdir(parents=True, exist_ok=True)
new_files = list(new_sources_dir.glob('*.csv')) + list(new_sources_dir.glob('*.xlsx'))

if new_files:
    new_parts = []
    for p in new_files:
        try:
            nd = pd.read_csv(p) if p.suffix.lower() == '.csv' else pd.read_excel(p)
            nd = map_to_standard_schema(nd)
            nd['source_file'] = str(p)
            new_parts.append(nd)
        except Exception as ex:
            print('Skip file', p, 'reason:', ex)

    if new_parts:
        new_raw = pd.concat(new_parts, ignore_index=True, sort=False)
        raw_extended = incremental_merge(raw_df, new_raw)

        # Внешние источники (заглушка): экономика/регион/отрасль
        ext_path = Path('data/external_sources/regional_indicators.csv')
        if ext_path.exists() and 'Область' in raw_extended.columns:
            ext = pd.read_csv(ext_path)
            raw_extended = raw_extended.merge(ext, on='Область', how='left')

        # Контроль качества и drift (простая версия)
        print('Base rows:', len(raw_df), 'Extended rows:', len(raw_extended))
        if 'Причитающая сумма' in raw_extended.columns:
            base_mean = raw_df['Причитающая сумма'].mean() if 'Причитающая сумма' in raw_df.columns else np.nan
            ext_mean = raw_extended['Причитающая сумма'].mean()
            print('Amount mean drift:', round((ext_mean - base_mean), 2))

        # Переобучение на расширенных данных
        df_ext = scoring.preprocess_data(raw_extended)
        df_ext, _ = scoring.create_kmeans_segments(df_ext, n_clusters=4)
        df_ext = scoring.detect_anomalies(df_ext)
        df_ext = scoring.calculate_risk_score(df_ext)
        df_ext[target_col] = (df_ext['Статус заявки'].astype(str) == 'Отклонена').astype(int)

        X_ext = df_ext[features].copy()
        y_ext = df_ext[target_col].copy()
        X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
            X_ext, y_ext, test_size=0.2, random_state=SEED, stratify=y_ext
        )

        best_model.fit(X_train_e, y_train_e)
        proba_e = best_model.predict_proba(X_test_e)[:, 1]
        pred_e = (proba_e >= 0.5).astype(int)

        print('\nMetrics on extended data:')
        print('ROC-AUC:', round(roc_auc_score(y_test_e, proba_e), 4))
        print('PR-AUC:', round(average_precision_score(y_test_e, proba_e), 4))
        print('F1:', round(f1_score(y_test_e, pred_e), 4))
        print('Recall rejected:', round(recall_score(y_test_e, pred_e), 4))

        # Сохранение артефактов
        Path('results').mkdir(exist_ok=True)
        df_ext.to_csv('results/processed_data_extended.csv', index=False, encoding='utf-8-sig')
        print('Saved: results/processed_data_extended.csv')
else:
    print('Добавьте файлы в data/new_sources, затем перезапустите эту ячейку.')